# Final Project: Baseline vs Transformer Models Comparison

This notebook loads saved metrics and predictions from both model families and produces a unified comparison with limitations analysis.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from sklearn.metrics import classification_report, confusion_matrix
import os

os.makedirs("plotly_graphs", exist_ok=True)

BLUE_PALETTE  = ["#084594","#2171b5","#4292c6","#6baed6","#9ecae1"]
GREEN_PALETTE = ["#005a32","#238b45","#41ab5d","#74c476","#a1d99b"]
MIXED_PALETTE = ["#2171b5","#41ab5d","#6baed6","#74c476","#084594","#238b45","#9ecae1"]


## 1. Load saved metrics from both model families

In [ ]:
df_kaggle = pd.read_csv(
    "kaggle_train_dataset.csv",
    engine="python", on_bad_lines="warn",
    encoding="utf-8", encoding_errors="replace"
)

label_map = (
    df_kaggle[["label", "category"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["category"]
    .to_dict()
)
label_names = [label_map[k] for k in sorted(label_map.keys())]
print("Label map:", label_map)


Label map: {0: 'crime', 1: 'economy', 2: 'education', 3: 'health', 4: 'international', 5: 'other', 6: 'politics', 7: 'social', 8: 'sports'}


In [ ]:
baseline_df = pd.read_csv("baseline_models_test_metrics.csv")
transformer_df = pd.read_csv("transformer_model_metrics.csv")

baseline_df["Family"] = "Baseline (TF-IDF)"
transformer_df["Family"] = "Transformer"

baseline_df = baseline_df.rename(columns={
    "Precision (mac)": "Precision",
    "Recall (mac)":    "Recall",
    "F1 (weighted)":   "F1-score"
})

cols = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "Family"]
baseline_df    = baseline_df[cols]
transformer_df = transformer_df[[c for c in cols if c in transformer_df.columns]]

all_models_df = pd.concat([baseline_df, transformer_df], ignore_index=True)
all_models_df = all_models_df.sort_values("F1-score", ascending=False).reset_index(drop=True)
all_models_df.to_csv("final_all_models_comparison.csv", index=False)
print(all_models_df[["Model", "Family", "F1-score"]].to_string(index=False))


                             Model            Family  F1-score
                        Linear SVM Baseline (TF-IDF)  0.767000
               Logistic Regression Baseline (TF-IDF)  0.765500
      bert-base-multilingual-cased       Transformer  0.756434
distilbert-base-multilingual-cased       Transformer  0.742100
                  xlm-roberta-base       Transformer  0.736886
            Complement Naive Bayes Baseline (TF-IDF)  0.707900


## 2. Model comparison — all models

In [ ]:
fig = px.bar(
    all_models_df,
    x="Model", y="F1-score",
    color="Family",
    text_auto=".3f",
    title="F1-score Comparison: Baseline vs Transformer Models",
    color_discrete_map={
        "Baseline (TF-IDF)": "#2171b5",
        "Transformer":       "#41ab5d"
    },
    barmode="group"
)
fig.update_layout(yaxis_range=[0.5, 1], width=1100, height=600)
fig.show()
fig.write_html("plotly_graphs/final_f1_comparison.html")


In [ ]:
metrics_long = all_models_df.melt(
    id_vars=["Model","Family"],
    value_vars=["Accuracy","Precision","Recall","F1-score"],
    var_name="Metric", value_name="Score"
)

fig = px.bar(
    metrics_long,
    x="Metric", y="Score",
    color="Model",
    text_auto=".3f",
    barmode="group",
    title="All Metrics: Baseline vs Transformer Models",
    color_discrete_sequence=MIXED_PALETTE
)
fig.update_layout(yaxis_range=[0.5, 1], width=1200, height=600)
fig.show()
fig.write_html("plotly_graphs/final_all_metrics_comparison.html")


In [ ]:
from plotly.graph_objects import Figure, Scatterpolar

metrics = ["Accuracy","Precision","Recall","F1-score"]
fig = Figure()

colors = BLUE_PALETTE[:len(baseline_df)] + GREEN_PALETTE[:len(transformer_df)]
for i, row in all_models_df.iterrows():
    vals = [row[m] for m in metrics] + [row[metrics[0]]]
    fig.add_trace(Scatterpolar(
        r=vals,
        theta=metrics + [metrics[0]],
        fill="toself",
        name=row["Model"],
        line_color=colors[i],
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0.5, 1])),
    title="Radar Chart — All Models",
    width=800, height=600
)
fig.show()
fig.write_html("plotly_graphs/final_radar_chart.html")


## 3. Inference predictions comparison

Compare what each model family predicted on the same Kazakhstan inference texts.

In [ ]:
inf_transformer = pd.read_csv("kazakhstan_inference_best_transformer_predictions.csv")
inf_baseline    = pd.read_csv("kazakhstan_inference_baseline_predictions.csv")

inf_transformer = inf_transformer.rename(columns={
    "predicted_category_transformer": "transformer_category",
    "predicted_label_transformer": "transformer_label"
})
inf_baseline = inf_baseline.rename(columns={
    "predicted_category": "baseline_category",
    "predicted_label":    "baseline_label"
})

inf_merged = inf_transformer[["input_text","transformer_category","transformer_label"]].merge(
    inf_baseline[["input_text","baseline_category","baseline_label"]],
    on="input_text", how="inner"
)

inf_merged["agreement"] = inf_merged["transformer_category"] == inf_merged["baseline_category"]

total = len(inf_merged)
agree = inf_merged["agreement"].sum()
print(f"Total inference samples : {total}")
print(f"Models agree            : {agree}  ({agree/total*100:.1f}%)")
print(f"Models disagree         : {total-agree}  ({(total-agree)/total*100:.1f}%)")
inf_merged.to_csv("inference_comparison_merged.csv", index=False)
inf_merged.head()


Total inference samples : 11513
Models agree            : 5232  (45.4%)
Models disagree         : 6281  (54.6%)


,input_text,transformer_category,transformer_label,baseline_category,baseline_label,agreement
0,"астана, алматы және шымкентте есірткі тұтынушы...",crime,0,education,2,False
1,“ешқандай таныс ықпал ете алмайды“ - вице-мини...,other,5,crime,0,False
2,“сіз iphone сатып алдыңыз“: қазақстандықтарға ...,crime,0,crime,0,True
3,астанада парламент палаталарының бірлескен оты...,politics,6,politics,6,True
4,астана мен алматыда 14-16 мамырда ауа райы қан...,other,5,international,4,False


In [ ]:
dist_transformer = inf_merged["transformer_category"].value_counts().reset_index()
dist_transformer.columns = ["Category","Count"]
dist_transformer["Model"] = "Transformer (best)"

dist_baseline = inf_merged["baseline_category"].value_counts().reset_index()
dist_baseline.columns = ["Category","Count"]
dist_baseline["Model"] = "Baseline (best)"

dist_combined = pd.concat([dist_transformer, dist_baseline], ignore_index=True)

fig = px.bar(
    dist_combined,
    x="Category", y="Count",
    color="Model",
    barmode="group",
    text_auto=True,
    title="Predicted Category Distribution — Transformer vs Baseline on Inference Data",
    color_discrete_map={
        "Transformer (best)": "#41ab5d",
        "Baseline (best)":    "#2171b5"
    }
)
fig.update_layout(width=1100, height=600)
fig.show()
fig.write_html("plotly_graphs/inference_distribution_comparison.html")


In [ ]:
disagree_df = inf_merged[~inf_merged["agreement"]].copy()

disagree_pairs = (
    disagree_df
    .groupby(["transformer_category","baseline_category"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
disagree_pairs["pair"] = disagree_pairs["transformer_category"] + " → " + disagree_pairs["baseline_category"]

fig = px.bar(
    disagree_pairs.head(15),
    x="count", y="pair",
    orientation="h",
    text_auto=True,
    title="Top 15 Disagreements: Transformer → Baseline on Inference Data",
    labels={"count": "Number of Samples", "pair": "Transformer → Baseline"},
    color_discrete_sequence=["#e6550d"]
)
fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=600)
fig.show()
fig.write_html("plotly_graphs/inference_disagreement_pairs.html")
print(disagree_pairs.head(15).to_string(index=False))


transformer_category baseline_category  count                     pair
               other             crime   1473            other → crime
               other          politics   1245         other → politics
               other     international    944    other → international
               other            health    706           other → health
               other           economy    524          other → economy
               other         education    327        other → education
               other            sports    106           other → sports
             economy          politics     98       economy → politics
               other            social     72           other → social
            politics             crime     66         politics → crime
              health             crime     45           health → crime
               crime          politics     44         crime → politics
              health          politics     40        health → politics
      

In [ ]:
print("=== EXAMPLE DISAGREEMENTS ===")
examples = disagree_df[["input_text","transformer_category","baseline_category"]].head(10)
for _, row in examples.iterrows():
    print(f"Text     : {row['input_text'][:200]}")
    print(f"Transformer: {row['transformer_category']}")
    print(f"Baseline   : {row['baseline_category']}")
    print()


=== EXAMPLE DISAGREEMENTS ===
Text     : астана, алматы және шымкентте есірткі тұтынушылар жаңа әдіспен анықталады астана, алматы және шымкентте есірткі тұтынушылар жаңа әдіспен анықталады қазақстанда ағын суларда есірткінің бар-жоғы зерттел
Transformer: crime
Baseline   : education

Text     : “ешқандай таныс ықпал ете алмайды“ - вице-министр қызылордадағы жантүршігерлік жол апаты туралы “ешқандай таныс ықпал ете алмайды“ - вице-министр қызылордадағы жантүршігерлік жол апаты туралы ішкі іст
Transformer: other
Baseline   : crime

Text     : астана мен алматыда 14-16 мамырда ауа райы қандай болады? астана мен алматыда 14-16 мамырда ауа райы қандай болады? "қазгидромет" синоптиктері астана мен алматыда 14, 15 және 16 мамырда ауа райы қанда
Transformer: other
Baseline   : international

Text     : декларация-2026. 15 қыркүйекке дейін есеп бермесе, кімдерге айыппұл салынады? декларация-2026. 15 қыркүйекке дейін есеп бермесе, кімдерге айыппұл салынады? 2026 жылы қазақстандықтардың бір бөліг

## 4. Limitations Analysis

In [ ]:
errors_transformer = pd.read_csv("error_analysis_misclassified.csv")
errors_baseline    = pd.read_csv("error_analysis_misclassified_baseline.csv")

errors_transformer["family"] = "Transformer"
errors_baseline["family"]    = "Baseline"

err_t = errors_transformer["true_category"].value_counts().rename("Transformer errors")
err_b = errors_baseline["true_category"].value_counts().rename("Baseline errors")

class_errors = pd.concat([err_t, err_b], axis=1).fillna(0).astype(int)
class_errors = class_errors.reset_index().rename(columns={"true_category": "Category"})
class_errors_long = class_errors.melt(id_vars="Category", var_name="Family", value_name="Error count")

fig = px.bar(
    class_errors_long,
    x="Category", y="Error count",
    color="Family",
    barmode="group",
    text_auto=True,
    title="Number of Errors per Class: Transformer vs Baseline",
    color_discrete_map={
        "Transformer errors": "#41ab5d",
        "Baseline errors":    "#2171b5"
    }
)
fig.update_layout(width=1100, height=550)
fig.show()
fig.write_html("plotly_graphs/limitations_errors_per_class.html")

In [ ]:
errors_transformer["text_length"] = errors_transformer["text"].str.len()
errors_baseline["text_length"]    = errors_baseline["text"].str.len()

bins       = [0, 100, 300, 600, 1000, 99999]
labels_bins = ["<100", "100-300", "300-600", "600-1000", ">1000"]

errors_transformer["length_bin"] = pd.cut(errors_transformer["text_length"], bins=bins, labels=labels_bins)
errors_baseline["length_bin"]    = pd.cut(errors_baseline["text_length"],    bins=bins, labels=labels_bins)

len_t = errors_transformer["length_bin"].value_counts().rename("Transformer").sort_index()
len_b = errors_baseline["length_bin"].value_counts().rename("Baseline").sort_index()

len_df = pd.concat([len_t, len_b], axis=1).reset_index().rename(columns={"length_bin": "Length bin"})  # ← исправлено
len_long = len_df.melt(id_vars="Length bin", var_name="Family", value_name="Error count")

fig = px.bar(
    len_long,
    x="Length bin", y="Error count",
    color="Family", barmode="group",
    text_auto=True,
    title="Errors by Text Length: Transformer vs Baseline",
    color_discrete_map={"Transformer": "#41ab5d", "Baseline": "#2171b5"}
)
fig.update_layout(width=900, height=500)
fig.show()
fig.write_html("plotly_graphs/limitations_text_length_errors.html")

In [ ]:
common_errors = pd.merge(
    errors_transformer[["text","true_category","pred_category"]],
    errors_baseline[["text","true_category","pred_category"]],
    on=["text","true_category"],
    suffixes=("_transformer","_baseline")
)
print(f"Examples BOTH models got wrong: {len(common_errors)}")
print()

hard_classes = common_errors["true_category"].value_counts().reset_index()
hard_classes.columns = ["Category","Both models wrong"]

fig = px.bar(
    hard_classes,
    x="Category", y="Both models wrong",
    text_auto=True,
    title="Classes Both Models Struggle With (Structural Data Problem)",
    color_discrete_sequence=["#e6550d"]
)
fig.update_layout(height=500)
fig.show()
fig.write_html("plotly_graphs/limitations_common_hard_classes.html")
print(hard_classes.to_string(index=False))


Examples BOTH models got wrong: 7



 Category  Both models wrong
  economy                  3
education                  3
   health                  1


Вот короткий текст для вставки в Google Colab:

---

## 5. Key Findings & Limitations

### 5.1 Model Performance

| Model | Family | Weighted F1 |
|---|---|---|
| Linear SVM | Baseline | ~0.767 |
| Logistic Regression | Baseline | ~0.766 |
| mBERT | Transformer | ~0.759 |
| DistilBERT | Transformer | ~0.745 |
| XLM-RoBERTa | Transformer | ~0.740 |
| Complement Naive Bayes | Baseline | ~0.708 |

Baseline models matched or slightly outperformed all transformer models. The gap is under 1% — the performance ceiling is set by **data quality**, not model capacity.

---

### 5.2 Inference — Agreement & Disagreement

- Models **agree** most on `sports` and `health` — distinctive vocabulary, no ambiguity
- Models **disagree** most on `other` vs `politics`, `other` vs `crime`, `economy` vs `politics` — semantically overlapping categories where TF-IDF and transformers make different assumptions
- Neither is provably correct without ground truth labels

---

### 5.3 Shared Limitations

**`other` class is the root cause of most errors** — 40% of data, semantically undefined, absorbs articles that don't fit other categories. Both families show `crime -> other` and `politics -> other` as top confused pairs. This is a labelling problem, not a modelling problem.

**`social` and `education` are structurally hard** — low support (<80 test examples each), shared vocabulary with `politics` and `economy`. Affects both model families equally.


